# PF4 Heart Failure Associations from HERMES GWAS

This notebook processes heart failure summary statistics from the HERMES dataset to identify SNP associations within the PF4 genomic region. Variants are filtered based on genomic coordinates and cross-referenced with PF4 SNPs from NCBI dbSNP.

In [1]:
import pandas as pd
import json
from pathlib import Path

pd.set_option("display.max_columns", None)

In [2]:
region_path = Path("../data/interim/ensembl/region.json")
variants_path = Path("../data/interim/ncbi/variants.csv")

hermes_path = Path("../data/raw/hermes/heart_failure.tsv")

out_dir = Path("../data/interim/hermes")
out_dir.mkdir(parents=True, exist_ok=True)

out_associations_csv = out_dir / "associations.csv"
out_summary_json = out_dir / "summary.json"

In [3]:
with open(region_path, "r") as f:
    pf4_region = json.load(f)

variants_df = pd.read_csv(variants_path)

print("Gene:", pf4_region["gene_symbol"])
print("Assembly:", pf4_region["assembly_name"])
print(
    "PF4 region:",
    f"chr{pf4_region['chromosome']}:{pf4_region['region_start']}-{pf4_region['region_end']}"
)
print("Number of PF4 variants:", variants_df.shape[0])

variants_df.head()

Gene: PF4
Assembly: GRCh38
PF4 region: chr4:73930811-74032027
Number of PF4 variants: 2030


,rsID,Variant_type,Alleles_REF_ALT,CHRPOS_GRCh38,CHRPOS_GRCh37,Canonical_SPDI,Gene_name,Functional_consequence,Clinical_significance,Validation_flags,ALFA_freq,ALFA_AC,ALSPAC_freq,ALSPAC_AC,TOMMO_freq,TOMMO_AC,MAX_freq,Priority,Position_GRCh38,Location_relative_to_PF4
0,rs2233648,snv,C>G,4:73981419,4:74847136,NC_000004.12:73981418:C:G,PF4,"coding_sequence_variant,synonymous_variant",benign,"by-frequency,by-alfa,by-cluster",0.005184,244.0,0.0,0.0,0.000039,3.0,0.005184,Primary,73981419,inside_PF4
1,rs367951530,snv,G>A;G>T,4:73981468,4:74847185,"NC_000004.12:73981467:G:A,NC_000004.12:7398146...",PF4,"coding_sequence_variant,missense_variant",uncertain-significance,"by-frequency,by-alfa,by-cluster",0.000058,3.0,NaN,NaN,NaN,NaN,0.000058,Below threshold,73981468,inside_PF4
2,rs764474600,snv,C>G;C>T,4:73981460,4:74847177,"NC_000004.12:73981459:C:G,NC_000004.12:7398145...",PF4,"coding_sequence_variant,missense_variant",uncertain-significance,"by-frequency,by-alfa,by-cluster",0.000000,0.0,NaN,NaN,0.000013,1.0,0.000013,Below threshold,73981460,inside_PF4
3,rs765704070,snv,C>G;C>T,4:73981445,4:74847162,"NC_000004.12:73981444:C:G,NC_000004.12:7398144...",PF4,"coding_sequence_variant,missense_variant",uncertain-significance,"by-frequency,by-alfa,by-cluster",0.000000,0.0,NaN,NaN,NaN,NaN,0.000000,Below threshold,73981445,inside_PF4
4,rs982761496,snv,C>A,4:73981500,4:74847217,NC_000004.12:73981499:C:A,PF4,"coding_sequence_variant,missense_variant",uncertain-significance,"by-frequency,by-alfa",0.000000,0.0,NaN,NaN,NaN,NaN,0.000000,Below threshold,73981500,inside_PF4


In [4]:
if not hermes_path.exists():
    raise FileNotFoundError(
        f"HERMES file not found at: {hermes_path}\n"
    )

In [5]:
hermes_preview = pd.read_csv(
    hermes_path,
    sep="\t",
    nrows=5
)

hermes_preview

,MarkerName,Allele1,Allele2,Freq1,FreqSE,MinFreq,MaxFreq,Effect,StdErr,P.value,Direction,HetISq,HetChiSq,HetDf,HetPVal,TotalSampleSize,nstudies,final_rsid,CHR,BP
0,1:100000012,t,g,0.2582,0.0208,0.2257,0.3068,0.0088,0.0099,0.3758,--+-?+-++-+-++++--+++-----++,0.0,17.928,26,0.8784,564738,27,rs10875231,1,100000012
1,1:100000827,t,c,0.3194,0.0295,0.2733,0.3910,0.0108,0.0094,0.2532,--+-?+++-++--++?+-+++-----++,0.0,20.478,25,0.7213,556506,26,rs6678176,1,100000827
2,1:100000843,t,c,0.9374,0.0098,0.9147,0.9538,-0.0150,0.0183,0.4124,-+++?---+---++++--?--++---+-,0.0,20.905,25,0.6979,559493,26,rs78286437,1,100000843
3,1:100000989:ID,d,i,0.9375,0.0131,0.9161,0.9588,-0.0296,0.0253,0.2416,-??----++?-???++--?--+?--?+?,13.8,19.712,17,0.2892,155499,18,rs146963890,1,100000989
4,1:100001138,a,g,0.9695,0.0076,0.9641,0.9880,0.0209,0.0334,0.5314,-????-+++??+-+++-??-+-?-+??+,0.0,13.664,16,0.6238,481945,17,rs144406489,1,100001138


In [6]:
pf4_rsids = (
    variants_df["rsID"]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)

priority_pf4_rsids = (
    variants_df[
        variants_df["Priority"].isin(["Primary", "Secondary"])
    ]["rsID"]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)

pf4_rsids_set = set(pf4_rsids)
priority_pf4_rsids_set = set(priority_pf4_rsids)

print("Total PF4 rsIDs:", len(pf4_rsids_set))
print("Priority PF4 rsIDs:", len(priority_pf4_rsids_set))

Total PF4 rsIDs: 1976
Priority PF4 rsIDs: 176


In [7]:
usecols = [
    "MarkerName",
    "Allele1",
    "Allele2",
    "Freq1",
    "FreqSE",
    "MinFreq",
    "MaxFreq",
    "Effect",
    "StdErr",
    "P.value",
    "Direction",
    "HetISq",
    "HetChiSq",
    "HetDf",
    "HetPVal",
    "TotalSampleSize",
    "nstudies",
    "final_rsid",
    "CHR",
    "BP"
]

matched_chunks = []

for chunk in pd.read_csv(
    hermes_path,
    sep="\t",
    usecols=usecols,
    chunksize=500_000
):
    chunk["final_rsid"] = chunk["final_rsid"].astype(str)

    matched = chunk[chunk["final_rsid"].isin(pf4_rsids_set)].copy()

    if not matched.empty:
        matched_chunks.append(matched)

if matched_chunks:
    hermes_matches_df = pd.concat(matched_chunks, ignore_index=True)
else:
    hermes_matches_df = pd.DataFrame(columns=usecols)

hermes_matches_df.shape

(5, 20)

In [8]:
hermes_matches_df = hermes_matches_df.rename(columns={
    "final_rsid": "rsID",
    "MarkerName": "HERMES_marker_name",
    "CHR": "HERMES_CHR",
    "BP": "HERMES_BP",
    "Allele1": "HERMES_Allele1",
    "Allele2": "HERMES_Allele2",
    "Freq1": "HERMES_Allele1_freq",
    "FreqSE": "HERMES_freq_se",
    "MinFreq": "HERMES_min_freq",
    "MaxFreq": "HERMES_max_freq",
    "Effect": "HERMES_effect",
    "StdErr": "HERMES_std_error",
    "P.value": "HERMES_p_value",
    "Direction": "HERMES_direction",
    "HetISq": "HERMES_het_i_sq",
    "HetChiSq": "HERMES_het_chi_sq",
    "HetDf": "HERMES_het_df",
    "HetPVal": "HERMES_het_p_value",
    "TotalSampleSize": "HERMES_total_sample_size",
    "nstudies": "HERMES_n_studies"
})

hermes_matches_df["phenotype"] = "Heart failure"
hermes_matches_df["source_dataset"] = "HERMES Heart Failure summary statistics without UK Biobank samples"

hermes_matches_df.head()

,HERMES_marker_name,HERMES_Allele1,HERMES_Allele2,HERMES_Allele1_freq,HERMES_freq_se,HERMES_min_freq,HERMES_max_freq,HERMES_effect,HERMES_std_error,HERMES_p_value,HERMES_direction,HERMES_het_i_sq,HERMES_het_chi_sq,HERMES_het_df,HERMES_het_p_value,HERMES_total_sample_size,HERMES_n_studies,rsID,HERMES_CHR,HERMES_BP,phenotype,source_dataset
0,4:74846661,a,c,0.0385,0.0083,0.0246,0.0562,-0.0121,0.0241,0.6149,-?-+-++----++++?+-----+-++--,10.2,27.854,25,0.3146,555920.0,26,rs11574452,4,74846661,Heart failure,HERMES Heart Failure summary statistics withou...
1,4:74847111,a,c,0.9607,0.0088,0.9399,0.9751,0.0147,0.0236,0.5315,+?+-+--++++----+-+++++-+--++,0.0,25.713,26,0.4790,564275.0,27,rs11574450,4,74847111,Heart failure,HERMES Heart Failure summary statistics withou...
2,4:74847292,c,g,0.8773,0.0150,0.8588,0.9192,-0.0036,0.0134,0.7903,++--?+++--+++---++---+---+++,0.0,20.006,26,0.7913,564832.0,27,rs352007,4,74847292,Heart failure,HERMES Heart Failure summary statistics withou...
3,4:74848065,c,g,0.0398,0.0092,0.0248,0.0601,-0.0156,0.0235,0.5076,-?-+-++----++++-+-----+-++--,0.0,25.603,26,0.4851,564275.0,27,rs3756074,4,74848065,Heart failure,HERMES Heart Failure summary statistics withou...
4,4:74849352,t,c,0.1199,0.0159,0.0769,0.1393,0.0050,0.0135,0.7106,+-+-?---++---+++--+++-+++---,0.0,21.673,26,0.7064,564852.0,27,rs394408,4,74849352,Heart failure,HERMES Heart Failure summary statistics withou...


In [9]:
numeric_cols = [
    "HERMES_CHR",
    "HERMES_BP",
    "HERMES_Allele1_freq",
    "HERMES_freq_se",
    "HERMES_min_freq",
    "HERMES_max_freq",
    "HERMES_effect",
    "HERMES_std_error",
    "HERMES_p_value",
    "HERMES_het_i_sq",
    "HERMES_het_chi_sq",
    "HERMES_het_df",
    "HERMES_het_p_value",
    "HERMES_total_sample_size",
    "HERMES_n_studies"
]

for col in numeric_cols:
    hermes_matches_df[col] = pd.to_numeric(
        hermes_matches_df[col],
        errors="coerce"
    )

hermes_matches_df = hermes_matches_df.sort_values(
    "HERMES_p_value",
    ascending=True
)

hermes_matches_df.head()

,HERMES_marker_name,HERMES_Allele1,HERMES_Allele2,HERMES_Allele1_freq,HERMES_freq_se,HERMES_min_freq,HERMES_max_freq,HERMES_effect,HERMES_std_error,HERMES_p_value,HERMES_direction,HERMES_het_i_sq,HERMES_het_chi_sq,HERMES_het_df,HERMES_het_p_value,HERMES_total_sample_size,HERMES_n_studies,rsID,HERMES_CHR,HERMES_BP,phenotype,source_dataset
3,4:74848065,c,g,0.0398,0.0092,0.0248,0.0601,-0.0156,0.0235,0.5076,-?-+-++----++++-+-----+-++--,0.0,25.603,26,0.4851,564275.0,27,rs3756074,4,74848065,Heart failure,HERMES Heart Failure summary statistics withou...
1,4:74847111,a,c,0.9607,0.0088,0.9399,0.9751,0.0147,0.0236,0.5315,+?+-+--++++----+-+++++-+--++,0.0,25.713,26,0.4790,564275.0,27,rs11574450,4,74847111,Heart failure,HERMES Heart Failure summary statistics withou...
0,4:74846661,a,c,0.0385,0.0083,0.0246,0.0562,-0.0121,0.0241,0.6149,-?-+-++----++++?+-----+-++--,10.2,27.854,25,0.3146,555920.0,26,rs11574452,4,74846661,Heart failure,HERMES Heart Failure summary statistics withou...
4,4:74849352,t,c,0.1199,0.0159,0.0769,0.1393,0.0050,0.0135,0.7106,+-+-?---++---+++--+++-+++---,0.0,21.673,26,0.7064,564852.0,27,rs394408,4,74849352,Heart failure,HERMES Heart Failure summary statistics withou...
2,4:74847292,c,g,0.8773,0.0150,0.8588,0.9192,-0.0036,0.0134,0.7903,++--?+++--+++---++---+---+++,0.0,20.006,26,0.7913,564832.0,27,rs352007,4,74847292,Heart failure,HERMES Heart Failure summary statistics withou...


In [10]:
variant_annotation_cols = [
    "rsID",
    "MAX_freq",
    "Priority",
    "Variant_type",
    "Alleles_REF_ALT",
    "CHRPOS_GRCh38",
    "CHRPOS_GRCh37",
    "Functional_consequence",
    "Clinical_significance",
    "Location_relative_to_PF4"
]

variant_metadata = variants_df[variant_annotation_cols].drop_duplicates()

hermes_annotated_df = hermes_matches_df.merge(
    variant_metadata,
    on="rsID",
    how="left"
)

hermes_annotated_df.head()

,HERMES_marker_name,HERMES_Allele1,HERMES_Allele2,HERMES_Allele1_freq,HERMES_freq_se,HERMES_min_freq,HERMES_max_freq,HERMES_effect,HERMES_std_error,HERMES_p_value,HERMES_direction,HERMES_het_i_sq,HERMES_het_chi_sq,HERMES_het_df,HERMES_het_p_value,HERMES_total_sample_size,HERMES_n_studies,rsID,HERMES_CHR,HERMES_BP,phenotype,source_dataset,MAX_freq,Priority,Variant_type,Alleles_REF_ALT,CHRPOS_GRCh38,CHRPOS_GRCh37,Functional_consequence,Clinical_significance,Location_relative_to_PF4
0,4:74848065,c,g,0.0398,0.0092,0.0248,0.0601,-0.0156,0.0235,0.5076,-?-+-++----++++-+-----+-++--,0.0,25.603,26,0.4851,564275.0,27,rs3756074,4,74848065,Heart failure,HERMES Heart Failure summary statistics withou...,0.117814,Primary,snv,G>C;G>T,4:73982348,4:74848065,"2KB_upstream_variant,upstream_transcript_variant",NaN,near_PF4
1,4:74847111,a,c,0.9607,0.0088,0.9399,0.9751,0.0147,0.0236,0.5315,+?+-+--++++----+-+++++-+--++,0.0,25.713,26,0.4790,564275.0,27,rs11574450,4,74847111,Heart failure,HERMES Heart Failure summary statistics withou...,0.115542,Primary,snv,A>C,4:73981394,4:74847111,intron_variant,NaN,inside_PF4
2,4:74846661,a,c,0.0385,0.0083,0.0246,0.0562,-0.0121,0.0241,0.6149,-?-+-++----++++?+-----+-++--,10.2,27.854,25,0.3146,555920.0,26,rs11574452,4,74846661,Heart failure,HERMES Heart Failure summary statistics withou...,0.117362,Primary,snv,C>A,4:73980944,4:74846661,3_prime_UTR_variant,NaN,inside_PF4
3,4:74849352,t,c,0.1199,0.0159,0.0769,0.1393,0.0050,0.0135,0.7106,+-+-?---++---+++--+++-+++---,0.0,21.673,26,0.7064,564852.0,27,rs394408,4,74849352,Heart failure,HERMES Heart Failure summary statistics withou...,0.141156,Primary,snv,T>A;T>C;T>G,4:73983635,4:74849352,"2KB_upstream_variant,upstream_transcript_variant",NaN,near_PF4
4,4:74847292,c,g,0.8773,0.0150,0.8588,0.9192,-0.0036,0.0134,0.7903,++--?+++--+++---++---+---+++,0.0,20.006,26,0.7913,564832.0,27,rs352007,4,74847292,Heart failure,HERMES Heart Failure summary statistics withou...,0.142260,Primary,snv,G>A;G>C;G>T,4:73981575,4:74847292,intron_variant,NaN,inside_PF4


In [11]:
hermes_annotated_df.to_csv(out_associations_csv, index=False)

print("Saved:", out_associations_csv)

Saved: ../data/interim/hermes/associations.csv


In [12]:
priority_matches_count = int(
    hermes_annotated_df["Priority"].isin(["Primary", "Secondary"]).sum()
)

summary = {
    "source_dataset": "HERMES Heart Failure summary statistics without UK Biobank samples",
    "phenotype": "Heart failure",
    "total_pf4_variants": int(len(pf4_rsids_set)),
    "priority_pf4_variants": int(len(priority_pf4_rsids_set)),
    "hermes_pf4_matches": int(len(hermes_annotated_df)),
    "hermes_priority_matches": priority_matches_count,
}

with open(out_summary_json, "w") as f:
    json.dump(summary, f, indent=4)

summary

{'source_file': '../data/raw/hermes/heart_failure.tsv',
 'source_dataset': 'HERMES Heart Failure summary statistics without UK Biobank samples',
 'phenotype': 'Heart failure',
 'total_pf4_variants': 1976,
 'priority_pf4_variants': 176,
 'hermes_pf4_matches': 5,
 'hermes_priority_matches': 5}